# Part 3 — Optimize thresholds (rules / ML / hybrid)

**Inputs:**
- full-timeline features via `src.transform.prepare_train_val_test()`
- rules helpers in `src/rules.py`
- trained XGBoost in `artifacts/models/` (reused; no retrain)

**Handoff to Part 4:** `artifacts/strategy/frozen_strategy.json`

Optimize on validation at **FP = $30 only**, then freeze. Do not re-tune for other FP costs here.

Previous: [`02_train_models.ipynb`](02_train_models.ipynb) · Next: [`04_test_evaluation.ipynb`](04_test_evaluation.ipynb)


In [ ]:
from pathlib import Path
import sys

# Project root (works when cwd is notebooks/ or project root)
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.paths import (
    DATA_DIR,
    MODELS_DIR,
    ARTIFACTS_DIR,
    STRATEGY_DIR,
    TRAIN_CSV,
    TEST_CSV,
    XGB_MODEL_PATH,
    LOGISTIC_MODEL_PATH,
    FROZEN_STRATEGY_PATH,
)

print("Project root:", ROOT)
print("Data dir:", DATA_DIR)
print("Models dir:", MODELS_DIR)


## Section 3 - Optimising the threshold for all strategies (Rules / ML model / Hybrid):


### Rules strategy:


In [ ]:
from itertools import product
import joblib
from src.transform import prepare_train_val_test, prepare_model_frame

# Optimize at FP=$30 only; robustness grid is applied later on frozen test predictions
OPT_FP_COST = 30
ROBUSTNESS_FP_COSTS = [10, 30, 50, 100]

df_train, df_val, df_test = prepare_train_val_test()
best_xgb = joblib.load(XGB_MODEL_PATH)

print("Loaded existing model:", XGB_MODEL_PATH)
print("Optimize at FP=$", OPT_FP_COST, sep="")
print("Robustness grid (Part 4):", ROBUSTNESS_FP_COSTS)
print(df_train.shape, df_val.shape, df_test.shape)


In [ ]:
def evaluate_rule_with_cost(rule, df, fp_cost=30):
    metrics = evaluate_rule(rule, df)

    total_fn_cost = df.loc[
        (~rule) & (df["is_fraud"] == 1),
        "amt"
    ].sum()

    metrics["total_cost"] = (
        metrics["fp"] * fp_cost +
        total_fn_cost
    )
    metrics["fp_cost"] = fp_cost

    return metrics


def build_rule_strategies(df, amt, minutes, tx_23, tx_60):
    rule2 = (
        (df["amt"] >= amt) &
        df["category"].isin([
            "shopping_net",
            "misc_net",
            "grocery_pos"
        ])
    )

    rule3 = (
        (df["time_since_last_transaction"] <= pd.Timedelta(minutes=minutes)) &
        (df["amt"] >= amt)
    )

    rule5 = df["transactions_prev_60min"] >= tx_60

    rule7 = (
        (df["transactions_prev_60min"] >= tx_23) &
        (df["hour_of_day"] == 23)
    )

    rule8 = df["prev_fraud_exist"] == 1

    rules_high_precision = (
        (rule2 & rule7) |
        (rule3 & rule7) |
        (rule2 & rule5) |
        (rule3 & rule5)
    )

    rules_balanced = (
        rules_high_precision |
        (rule2 & rule3)
    )

    rules_high_recall = (
        rules_balanced |
        (rule2 & rule8)
    )

    return {
        "high_precision": rules_high_precision,
        "balanced": rules_balanced,
        "high_recall": rules_high_recall,
    }


In [ ]:
rule_results = []

for amt, minutes, tx_23, tx_60 in product(
    amt_candidates,
    minutes_between_transaction,
    num_transaction_at_23,
    num_transaction_60_min
):
    strategies = build_rule_strategies(
        df_val, amt, minutes, tx_23, tx_60
    )

    for strategy_name, rule in strategies.items():
        base_metrics = evaluate_rule(rule, df_val)
        total_fn_cost = df_val.loc[
            (~rule) & (df_val["is_fraud"] == 1),
            "amt"
        ].sum()

        for fp_cost in fp_costs:
            result = dict(base_metrics)
            result.update({
                "total_cost": base_metrics["fp"] * fp_cost + total_fn_cost,
                "fp_cost": fp_cost,
                "strategy": strategy_name,
                "amt_threshold": amt,
                "minutes_threshold": minutes,
                "transactions_23_threshold": tx_23,
                "transactions_60min_threshold": tx_60,
            })
            rule_results.append(result)

print(f"Evaluated {len(rule_results)} rule configurations across FP costs")


In [ ]:
rule_results = pd.DataFrame(rule_results)

best_rules_by_fp_cost = (
    rule_results
    .sort_values("total_cost")
    .groupby("fp_cost", as_index=False)
    .first()
)

best_rules_by_fp_cost[
    [
        "fp_cost",
        "strategy",
        "amt_threshold",
        "minutes_threshold",
        "transactions_23_threshold",
        "transactions_60min_threshold",
        "total_cost",
        "precision",
        "recall",
        "tp",
        "fp",
        "fn",
    ]
]


In [ ]:

rule_results[rule_results["fp_cost"] == 30].sort_values(
    "total_cost", ascending=True
).head(10)


In [ ]:
rule_results[
    (rule_results["fp_cost"] == 30) &
    (rule_results["strategy"] == "high_precision")
].sort_values("total_cost", ascending=True).head(10)


In [ ]:
# Best overall rules-only config per FP cost (any strategy)
# Plus best high-recall config per FP cost for the hybrid rule branch
best_rule_masks_val = {}
best_rule_masks_test = {}
high_recall_masks_val = {}
high_recall_masks_test = {}

best_high_recall_by_fp_cost = (
    rule_results[rule_results["strategy"] == "high_recall"]
    .sort_values("total_cost")
    .groupby("fp_cost", as_index=False)
    .first()
)

for _, row in best_rules_by_fp_cost.iterrows():
    fp_cost = int(row["fp_cost"])
    params = dict(
        amt=row["amt_threshold"],
        minutes=row["minutes_threshold"],
        tx_23=row["transactions_23_threshold"],
        tx_60=row["transactions_60min_threshold"],
    )
    val_strategies = build_rule_strategies(df_val, **params)
    test_strategies = build_rule_strategies(df_test, **params)
    strategy_name = row["strategy"]
    best_rule_masks_val[fp_cost] = val_strategies[strategy_name]
    best_rule_masks_test[fp_cost] = test_strategies[strategy_name]

for _, row in best_high_recall_by_fp_cost.iterrows():
    fp_cost = int(row["fp_cost"])
    params = dict(
        amt=row["amt_threshold"],
        minutes=row["minutes_threshold"],
        tx_23=row["transactions_23_threshold"],
        tx_60=row["transactions_60min_threshold"],
    )
    high_recall_masks_val[fp_cost] = build_rule_strategies(
        df_val, **params
    )["high_recall"]
    high_recall_masks_test[fp_cost] = build_rule_strategies(
        df_test, **params
    )["high_recall"]

# Keep the $30 masks under the old names for continuity
row30 = best_high_recall_by_fp_cost.loc[
    best_high_recall_by_fp_cost["fp_cost"] == 30
].iloc[0]

rules_at_30 = build_rule_strategies(
    df_val,
    amt=row30["amt_threshold"],
    minutes=row30["minutes_threshold"],
    tx_23=row30["transactions_23_threshold"],
    tx_60=row30["transactions_60min_threshold"],
)

rules_high_precision = rules_at_30["high_precision"]
rules_high_recall = high_recall_masks_val[30]
best_rules = rules_high_recall

display(best_rules_by_fp_cost)
display(best_high_recall_by_fp_cost)


In [ ]:
print("Best rules selected per FP cost")
for _, row in best_rules_by_fp_cost.iterrows():
    print(
        f"FP=${int(row['fp_cost'])}: {row['strategy']} | "
        f"amt>={row['amt_threshold']}, minutes<={row['minutes_threshold']}, "
        f"tx23>={row['transactions_23_threshold']}, "
        f"tx60>={row['transactions_60min_threshold']} | "
        f"val_cost={row['total_cost']:.2f}"
    )


## Final Rule-Strategy Selection

Rules were re-optimized on the validation set for each false-positive cost in `[10, 20, 30, 50, 100]`.

For each FP cost, the lowest-cost rule configuration (strategy + thresholds) was frozen and carried into the hybrid / test evaluation for that same cost.


## Optimising thresholds of ML models:


In [ ]:
def evaluate_model_threshold(y_true, probabilities, amounts, threshold, fp_cost=30):
    predictions = probabilities >= threshold

    tp = ((predictions == 1) & (y_true == 1)).sum()
    fp = ((predictions == 1) & (y_true == 0)).sum()
    fn = ((predictions == 0) & (y_true == 1)).sum()
    tn = ((predictions == 0) & (y_true == 0)).sum()

    fn_cost = amounts[
        (predictions == 0) & (y_true == 1)
    ].sum()

    total_cost = fp * fp_cost + fn_cost

    return {
        "threshold": threshold,
        "fp_cost": fp_cost,
        "total_cost": total_cost,
        "precision": tp / (tp + fp) if tp + fp else 0,
        "recall": tp / (tp + fn) if tp + fn else 0,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }


In [ ]:

thresholds = np.linspace(0.001, 0.40, 80)

xgb_val_prob = best_xgb.predict_proba(X_val)[:, 1]
val_amounts = df_val["amt"].to_numpy()

xgb_threshold_results = []

for fp_cost in fp_costs:
    for threshold in thresholds:
        xgb_threshold_results.append(
            evaluate_model_threshold(
                y_true=y_val,
                probabilities=xgb_val_prob,
                amounts=val_amounts,
                threshold=threshold,
                fp_cost=fp_cost,
            )
        )

xgb_threshold_results_df = pd.DataFrame(xgb_threshold_results)

best_xgb_threshold_by_fp = (
    xgb_threshold_results_df
    .sort_values("total_cost")
    .groupby("fp_cost", as_index=False)
    .first()
)


best_xgb_threshold = best_xgb_threshold_by_fp.loc[
    best_xgb_threshold_by_fp["fp_cost"] == 30
].iloc[0]

best_xgb_threshold_by_fp


In [ ]:

logistic_threshold_results = []
logistic_val_prob = logistic_grid_search.best_estimator_.predict_proba(X_val)[:, 1]

for threshold in thresholds:
    logistic_threshold_results.append(
        evaluate_model_threshold(
            y_true=y_val,
            probabilities=logistic_val_prob,
            amounts=val_amounts,
            threshold=threshold,
            fp_cost=30,
        )
    )

logistic_threshold_results_df = pd.DataFrame(logistic_threshold_results)
best_logistic_threshold = logistic_threshold_results_df.loc[
    logistic_threshold_results_df["total_cost"].idxmin()
]
best_logistic_threshold


In [ ]:
best_xgb_threshold_value = float(best_xgb_threshold["threshold"])
best_xgb_threshold_value


## Machine-Learning Model Selection

Decision thresholds for both machine-learning models were re-optimized on the validation set for each false-positive cost in `[10, 20, 30, 50, 100]`. The **$30** case below remains the highlighted baseline.

### Logistic Regression

The cost-minimizing Logistic Regression threshold was **0.081**.

At this threshold, the model achieved:

- **Total cost:** $274,380.30
- **Precision:** 32.53%
- **Recall:** 56.18%
- **True positives:** 864
- **False positives:** 1,792
- **False negatives:** 674
- **True negatives:** 256,005

### XGBoost

The cost-minimizing XGBoost threshold was **0.108**.

At this threshold, the model achieved:

- **Total cost:** $61,649.07
- **Precision:** 62.15%
- **Recall:** 91.29%
- **True positives:** 1,404
- **False positives:** 855
- **False negatives:** 134
- **True negatives:** 256,942

### Final Model Selection

XGBoost substantially outperformed Logistic Regression. It achieved a much lower total cost, higher precision, higher recall, fewer false positives, and far fewer missed fraud cases.

Therefore, **XGBoost was selected as the final machine-learning model**, using a probability threshold of **0.108**.


### Optimising thresholds of  Hybrid strategies:


In [ ]:
def evaluate_predictions(y_true, predictions, amounts, fp_cost=30):
    tp = ((predictions == 1) & (y_true == 1)).sum()
    fp = ((predictions == 1) & (y_true == 0)).sum()
    fn = ((predictions == 0) & (y_true == 1)).sum()
    tn = ((predictions == 0) & (y_true == 0)).sum()

    missed_fraud_cost = amounts[
        (predictions == 0) & (y_true == 1)
    ].sum()

    return {
        "fp_cost": fp_cost,
        "total_cost": fp * fp_cost + missed_fraud_cost,
        "precision": tp / (tp + fp) if tp + fp else 0,
        "recall": tp / (tp + fn) if tp + fn else 0,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }


In [ ]:
# Hybrid 1: model OR high-precision rules 
hybrid_1_results = []

for threshold in thresholds:
    hybrid_predictions = (
        (xgb_val_prob >= threshold) |
        rules_high_precision
    )

    result = evaluate_predictions(
        y_true=y_val,
        predictions=hybrid_predictions,
        amounts=df_val["amt"],
        fp_cost=30,
    )
    result["threshold"] = threshold
    hybrid_1_results.append(result)

hybrid_1_results_df = pd.DataFrame(hybrid_1_results)
best_hybrid_1 = hybrid_1_results_df.loc[
    hybrid_1_results_df["total_cost"].idxmin()
]
best_hybrid_1


In [ ]:
# Hybrid 2: fixed high threshold from $30 XGB optimum + rule-supported low threshold
hybrid_2_results = []
high_threshold = float(best_xgb_threshold_value)

for low_threshold in thresholds:
    if low_threshold >= high_threshold:
        continue

    hybrid_predictions = (
        (xgb_val_prob >= high_threshold) |
        (
            (xgb_val_prob >= low_threshold) &
            rules_high_recall
        )
    )

    result = evaluate_predictions(
        y_true=y_val,
        predictions=hybrid_predictions,
        amounts=df_val["amt"],
        fp_cost=30,
    )
    result["high_threshold"] = high_threshold
    result["low_threshold"] = low_threshold
    hybrid_2_results.append(result)

hybrid_2_results_df = pd.DataFrame(hybrid_2_results)
best_hybrid_2 = hybrid_2_results_df.loc[
    hybrid_2_results_df["total_cost"].idxmin()
]
best_hybrid_2


In [ ]:
# Hybrid 3: re-optimize high/low thresholds for each FP cost (coarse grid)
high_thresholds = np.linspace(0.02, 0.30, 40)
low_thresholds = np.linspace(0.001, 0.10, 30)

hybrid_3_results = []

for fp_cost in fp_costs:
    rules_mask = high_recall_masks_val[fp_cost]

    for high_threshold in high_thresholds:
        for low_threshold in low_thresholds:
            if low_threshold >= high_threshold:
                continue

            hybrid_predictions = (
                (xgb_val_prob >= high_threshold)
                | ((xgb_val_prob >= low_threshold) & rules_mask)
            )

            result = evaluate_predictions(
                y_true=y_val,
                predictions=hybrid_predictions,
                amounts=df_val["amt"],
                fp_cost=fp_cost,
            )
            result["high_threshold"] = high_threshold
            result["low_threshold"] = low_threshold
            hybrid_3_results.append(result)

hybrid_3_results_df = pd.DataFrame(hybrid_3_results)

best_hybrid_3_by_fp = (
    hybrid_3_results_df
    .sort_values("total_cost")
    .groupby("fp_cost", as_index=False)
    .first()
)

best_hybrid_3 = best_hybrid_3_by_fp.loc[
    best_hybrid_3_by_fp["fp_cost"] == 30
].iloc[0]

best_hybrid_3_by_fp


In [ ]:
# Validation comparison at $30 
hybrid_comparison_df = pd.DataFrame([
    {
        "strategy": "XGBoost only",
        **best_xgb_threshold.to_dict(),
    },
    {
        "strategy": "Hybrid 1: Model OR high-precision rules",
        **best_hybrid_1.to_dict(),
    },
    {
        "strategy": "Hybrid 2: Fixed high threshold + rule-supported low threshold",
        **best_hybrid_2.to_dict(),
    },
    {
        "strategy": "Hybrid 3: Optimized high and low thresholds",
        **best_hybrid_3.to_dict(),
    },
])


In [ ]:
hybrid_comparison_df = hybrid_comparison_df.sort_values(
    "total_cost"
).reset_index(drop=True)

hybrid_comparison_df


In [ ]:
# Validation summary across all FP costs: rules vs XGB vs Hybrid 3
val_strategy_rows = []

for fp_cost in fp_costs:
    rules_row = best_rules_by_fp_cost.loc[
        best_rules_by_fp_cost["fp_cost"] == fp_cost
    ].iloc[0]
    xgb_row = best_xgb_threshold_by_fp.loc[
        best_xgb_threshold_by_fp["fp_cost"] == fp_cost
    ].iloc[0]
    hybrid_row = best_hybrid_3_by_fp.loc[
        best_hybrid_3_by_fp["fp_cost"] == fp_cost
    ].iloc[0]

    val_strategy_rows.extend([
        {
            "fp_cost": fp_cost,
            "strategy": "Rules only",
            "total_cost": rules_row["total_cost"],
            "precision": rules_row["precision"],
            "recall": rules_row["recall"],
            "threshold": np.nan,
            "high_threshold": np.nan,
            "low_threshold": np.nan,
        },
        {
            "fp_cost": fp_cost,
            "strategy": "XGBoost only",
            "total_cost": xgb_row["total_cost"],
            "precision": xgb_row["precision"],
            "recall": xgb_row["recall"],
            "threshold": xgb_row["threshold"],
            "high_threshold": np.nan,
            "low_threshold": np.nan,
        },
        {
            "fp_cost": fp_cost,
            "strategy": "Hybrid",
            "total_cost": hybrid_row["total_cost"],
            "precision": hybrid_row["precision"],
            "recall": hybrid_row["recall"],
            "threshold": np.nan,
            "high_threshold": hybrid_row["high_threshold"],
            "low_threshold": hybrid_row["low_threshold"],
        },
    ])

val_strategy_comparison_df = pd.DataFrame(val_strategy_rows)
val_strategy_comparison_df


## Hybrid Strategy Selection

Hybrid thresholds were re-optimized on the validation set for each false-positive cost in `[10, 20, 30, 50, 100]`.

For each FP cost, Hybrid 3 searches a high XGBoost threshold and a lower rule-supported threshold using the high-recall rule family selected for that cost.

The **$30** baseline comparison against XGBoost-only / Hybrid 1 / Hybrid 2 is retained.
